# HAG/CAG Neuronal Marker Enrichment — Panels C and D

Reproduces panels C and D from `hag_cag_specificity_marker_enrichment.png`.

**Data source**: Trevino et al. snMultiome (CellXGene collection `ad2149fc`), 232,328 nuclei, 33 cell types.  
H5AD file expected at `/tmp/trevino_cortex.h5ad` (2.78 GB).  
Gene group file expected at `/mnt/user-uploads/HAG_CAG_NAG_For_Biomni.csv`.

**Panel C**: Odds ratio (HAG vs NAG, CAG vs NAG) for top-K neuronal marker gene enrichment across K = 5, 10, 15, 25, 50, 75, 100.  
**Panel D**: Per-neuronal-cell-type HAG marker enrichment (top-25 FC markers, two-sided Fisher's exact test, BH-FDR).

In [ ]:
import scanpy as sc
import numpy as np
import pandas as pd
from scipy.sparse import issparse
from scipy.stats import fisher_exact
from statsmodels.stats.multitest import multipletests
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')
import os
import subprocess
sns.set_theme(style='ticks', font_scale=1.05)

In [ ]:
adata = sc.read_h5ad('7b582c37-37ad-49d2-a369-06f38289031e.h5ad')


In [ ]:
adata.uns

## 1. Load data and define gene groups

In [ ]:
# Load H5AD and rebuild log-normalised AnnData from raw counts
print('Loading H5AD...')
adata = sc.read_h5ad('7b582c37-37ad-49d2-a369-06f38289031e.h5ad')
adata_norm = sc.AnnData(
    X=adata.raw.X.copy(),
    obs=adata.obs.copy(),
    var=adata.raw.var.copy()
)
adata_norm.var.index = adata.raw.var['feature_name'].values.astype(str)
adata_norm.var.index = adata_norm.var.index.astype(str)
adata_norm.var_names_make_unique()
sc.pp.normalize_total(adata_norm, target_sum=1e4)
sc.pp.log1p(adata_norm)
print(f'Normalised AnnData: {adata_norm.shape}')

In [ ]:
# Define HAG / CAG / NAG gene groups
# HAG: Z-score Difference >= 1.96 (human-accelerated)
# CAG: Z-score Difference <= -1.96 (chimpanzee-accelerated)
# NAG: |Z-score Difference| < 1.96 (not accelerated)
all_genes_df = pd.read_csv('HAG_CAG_NAG_For_Biomni.csv')
z = all_genes_df['Z-score Difference']
sig_mask = z.abs() >= 1.96
all_genes_df['group3'] = 'NAG'
all_genes_df.loc[sig_mask & (z > 0), 'group3'] = 'HAG'
all_genes_df.loc[sig_mask & (z < 0), 'group3'] = 'CAG'

sc_genes    = set(adata_norm.var_names)
hag_found   = [g for g in all_genes_df[all_genes_df['group3']=='HAG']['NearestGene'] if g in sc_genes]
cag_found   = [g for g in all_genes_df[all_genes_df['group3']=='CAG']['NearestGene'] if g in sc_genes]
nag_found   = [g for g in all_genes_df[all_genes_df['group3']=='NAG']['NearestGene'] if g in sc_genes]
gene_to_idx = {g: i for i, g in enumerate(adata_norm.var_names)}

study_genes = hag_found + cag_found + nag_found
study_idxs  = [gene_to_idx[g] for g in study_genes]
group_arr   = np.array(['HAG']*len(hag_found) + ['CAG']*len(cag_found) + ['NAG']*len(nag_found))

print(f'HAG={len(hag_found)}, CAG={len(cag_found)}, NAG={len(nag_found)}')

## 2. Define cell type sets

In [ ]:
cell_type_col  = 'Type_updated'
cell_types_all = sorted([ct for ct in adata_norm.obs[cell_type_col].unique() if ct != 'Unknown'])

# Neuronal cell types: EN-*, IN-*, Cajal-Retzius, EN-Newborn, IPC-EN, Tri-IPC
neuron_cell_types = [ct for ct in cell_types_all if (
    ct.startswith('EN-') or ct.startswith('IN-') or
    ct in ('Cajal-Retzius cell', 'EN-Newborn', 'IPC-EN', 'Tri-IPC')
)]
print(f'All cell types: {len(cell_types_all)}, Neuronal: {len(neuron_cell_types)}')
print('Neuronal cell types:', neuron_cell_types)

## 3. Build mean expression matrix (study genes x cell types)

In [ ]:
print(f'Building mean expression matrix: {len(study_genes)} genes x {len(cell_types_all)} cell types...')
mean_expr_all = np.zeros((len(study_genes), len(cell_types_all)))
for j, ct in enumerate(cell_types_all):
    mask = (adata_norm.obs[cell_type_col] == ct).values
    X_ct = adata_norm.X[mask, :]
    if issparse(X_ct):
        X_ct = X_ct.toarray()
    mean_expr_all[:, j] = X_ct[:, study_idxs].mean(axis=0)
    if (j + 1) % 10 == 0:
        print(f'  {j+1}/{len(cell_types_all)} done')

# Subset to neuronal cell types
neuron_idxs     = [cell_types_all.index(ct) for ct in neuron_cell_types]
mean_expr_neuro = mean_expr_all[:, neuron_idxs]
print(f'Neuron-only matrix: {mean_expr_neuro.shape}')

In [ ]:
mean_expr_neuro

## 4. Helper functions

In [ ]:
def compute_fc_matrix(mean_mat):
    """
    For each gene and cell type, compute fold-change over the mean of all
    other cell types (+ 0.1 pseudocount to stabilise low-expression genes).
    Returns FC matrix of same shape as mean_mat.
    """
    n_genes, n_ct = mean_mat.shape
    fc_mat = np.zeros_like(mean_mat)
    for j in range(n_ct):
        other_mean = (mean_mat.sum(axis=1) - mean_mat[:, j]) / (n_ct - 1)
        fc_mat[:, j] = mean_mat[:, j] / (other_mean + 0.1)
    return fc_mat


def marker_enrichment_test(fc_mat, group_arr, ct_names, top_k=25):
    """
    For each cell type, identify the top-K genes by FC.
    A gene is a 'marker' if it appears in the top-K of at least one cell type.
    Fisher's exact test is two-sided throughout.

    Returns:
      results  : dict with keys HAG_vs_NAG, CAG_vs_NAG, HAG_vs_CAG
      is_marker: bool array (n_genes,)
      best_rank: float array (n_genes,) -- best rank across all cell types
    """
    n_genes   = len(group_arr)
    is_marker = np.zeros(n_genes, dtype=bool)
    best_rank = np.full(n_genes, np.inf)

    for j in range(fc_mat.shape[1]):
        ranks = fc_mat[:, j].argsort()[::-1]   # descending FC
        is_marker[ranks[:top_k]] = True
        gene_ranks = np.empty(n_genes)
        gene_ranks[ranks] = np.arange(1, n_genes + 1)
        best_rank = np.minimum(best_rank, gene_ranks)

    results = {}
    for g1, g2 in [('HAG', 'NAG'), ('CAG', 'NAG'), ('HAG', 'CAG')]:
        m1 = is_marker[group_arr == g1]
        
        m2 = is_marker[group_arr == g2]
        a, b = int(m1.sum()), int((~m1).sum())
        c, d = int(m2.sum()), int((~m2).sum())
        or_, p = fisher_exact([[a, b], [c, d]], alternative='two-sided')
        results[f'{g1}_vs_{g2}'] = {
            'n_marker_g1': a, 'n_total_g1': int(len(m1)), 'pct_g1': m1.mean() * 100,
            'n_marker_g2': c, 'n_total_g2': int(len(m2)), 'pct_g2': m2.mean() * 100,
            'OR': or_, 'p': p,
        }
    return results, is_marker, best_rank


# Compute FC matrix for neuronal cell types
fc_neuro = compute_fc_matrix(mean_expr_neuro)
print('FC matrix computed.')

In [ ]:
x = pd.DataFrame(fc_neuro)
x.index = study_genes
x.columns = neuron_cell_types
x.to_csv("Cortical_Markers_Saved.csv")

## 5. Panel C data — OR across top-K thresholds (neurons)

In [ ]:
k_vals = [5, 10, 15, 25, 50, 75, 100]
panel_c_rows = []
for top_k in k_vals:
    res, _, _ = marker_enrichment_test(fc_neuro, group_arr, neuron_cell_types, top_k=top_k)
    for comp_key, g1, g2 in [
        ('HAG_vs_NAG', 'HAG', 'NAG'),
        ('CAG_vs_NAG', 'CAG', 'NAG'),
        ('HAG_vs_CAG', 'HAG', 'CAG'),
    ]:
        v = res[comp_key]
        panel_c_rows.append({
            'top_K':          top_k,
            'comparison':     comp_key,
            'group1':         g1,
            'group2':         g2,
            'n_marker_g1':    v['n_marker_g1'],
            'n_nonmarker_g1': v['n_total_g1'] - v['n_marker_g1'],
            'n_total_g1':     v['n_total_g1'],
            'pct_marker_g1':  round(v['pct_g1'], 2),
            'n_marker_g2':    v['n_marker_g2'],
            'n_nonmarker_g2': v['n_total_g2'] - v['n_marker_g2'],
            'n_total_g2':     v['n_total_g2'],
            'pct_marker_g2':  round(v['pct_g2'], 2),
            'odds_ratio':     round(v['OR'], 4),
            'p_value':        round(v['p'], 6),
        })

panel_c_df = pd.DataFrame(panel_c_rows)
panel_c_df.to_csv('panel_C_marker_OR_by_topK_neurons_bleh.csv', index=False)
print(panel_c_df.to_string(index=False))

## 6. Panel D data — per-cell-type HAG marker enrichment (top-25, neurons)

In [ ]:
top_k = 25
hag_idxs_study = np.where(group_arr == 'HAG')[0]
cag_idxs_study = np.where(group_arr == 'CAG')[0]
nag_idxs_study = np.where(group_arr == 'NAG')[0]

per_ct_results = []
for j, ct in enumerate(neuron_cell_types):
    fc_col       = fc_neuro[:, j]
    ranks        = fc_col.argsort()[::-1]
    top_gene_set = set(ranks[:top_k])

    n_hag = len(set(hag_idxs_study) & top_gene_set)
    n_cag = len(set(cag_idxs_study) & top_gene_set)
    n_nag = len(set(nag_idxs_study) & top_gene_set)

    # Two-sided Fisher: HAG vs NAG
    a, b   = n_hag, len(hag_idxs_study) - n_hag
    c, d   = n_nag, len(nag_idxs_study) - n_nag
    or_hag, p_hag = fisher_exact([[a, b], [c, d]], alternative='two-sided')

    # Two-sided Fisher: CAG vs NAG
    a2, b2 = n_cag, len(cag_idxs_study) - n_cag
    or_cag, p_cag = fisher_exact([[a2, b2], [c, d]], alternative='two-sided')

    top_hag_genes = [study_genes[i] for i in ranks[:top_k] if i in set(hag_idxs_study)]

    per_ct_results.append({
        'cell_type':       ct,
        'n_HAG_markers':   n_hag,
        'n_HAG_total':     len(hag_idxs_study),
        'pct_HAG':         round(n_hag / len(hag_idxs_study) * 100, 2),
        'n_CAG_markers':   n_cag,
        'n_CAG_total':     len(cag_idxs_study),
        'pct_CAG':         round(n_cag / len(cag_idxs_study) * 100, 2),
        'n_NAG_markers':   n_nag,
        'n_NAG_total':     len(nag_idxs_study),
        'pct_NAG':         round(n_nag / len(nag_idxs_study) * 100, 2),
        'OR_HAG_vs_NAG':   round(or_hag, 4),
        'p_HAG_vs_NAG':    round(p_hag, 6),
        'OR_CAG_vs_NAG':   round(or_cag, 4),
        'p_CAG_vs_NAG':    round(p_cag, 6),
        'top_HAG_genes':   ', '.join(top_hag_genes),
    })

per_ct_df = pd.DataFrame(per_ct_results)

# BH-FDR correction across the 22 per-cell-type tests
_, padj_hag, _, _ = multipletests(per_ct_df['p_HAG_vs_NAG'], method='fdr_bh')
_, padj_cag, _, _ = multipletests(per_ct_df['p_CAG_vs_NAG'], method='fdr_bh')
per_ct_df['p_adj_HAG_vs_NAG'] = padj_hag.round(6)
per_ct_df['p_adj_CAG_vs_NAG'] = padj_cag.round(6)

per_ct_df.to_csv('/mnt/results/hag_cag_neuronal_marker_enrichment_per_celltype.csv', index=False)
print(per_ct_df[[
    'cell_type', 'n_HAG_markers', 'pct_HAG', 'pct_NAG',
    'OR_HAG_vs_NAG', 'p_HAG_vs_NAG', 'p_adj_HAG_vs_NAG', 'top_HAG_genes'
]].sort_values('OR_HAG_vs_NAG', ascending=False).to_string(index=False))

## 7. Plot Panel C

In [ ]:
fig, ax = plt.subplots(figsize=(6, 4.5))

hag_rows = panel_c_df[panel_c_df['comparison'] == 'HAG_vs_NAG']
cag_rows = panel_c_df[panel_c_df['comparison'] == 'CAG_vs_NAG']

ax.plot(range(len(k_vals)), hag_rows['odds_ratio'].values,
        'o-', color='#E05C2A', linewidth=2, markersize=7, label='HAG vs NAG')
ax.plot(range(len(k_vals)), cag_rows['odds_ratio'].values,
        's--', color='#4A90D9', linewidth=2, markersize=7, label='CAG vs NAG')
ax.axhline(1.0, color='#AAAAAA', linewidth=1, linestyle=':')

# Significance stars above each point
for i, (ph, pc) in enumerate(zip(hag_rows['p_value'].values, cag_rows['p_value'].values)):
    if ph < 0.001:
        ax.text(i, hag_rows['odds_ratio'].values[i] + 0.18, '***',
                ha='center', fontsize=9, color='#E05C2A')
    elif ph < 0.05:
        ax.text(i, hag_rows['odds_ratio'].values[i] + 0.18, '*',
                ha='center', fontsize=9, color='#E05C2A')
    if pc < 0.05:
        ax.text(i, cag_rows['odds_ratio'].values[i] + 0.18, '*',
                ha='center', fontsize=9, color='#4A90D9')

ax.set_xticks(range(len(k_vals)))
ax.set_xticklabels([str(k) for k in k_vals])
ax.set_xlabel('Top-K markers per cell type', fontsize=11)
ax.set_ylabel('Odds ratio vs NAG', fontsize=11)
ax.set_title('Panel C: Marker enrichment OR\n(22 neuronal cell types, * p<0.05, *** p<0.001)',
             fontsize=11, fontweight='bold')
ax.legend(fontsize=10)
sns.despine(ax=ax)
plt.tight_layout()
plt.savefig('/mnt/results/panel_C_only.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved panel_C_only.png')

## 8. Plot Panel D

In [ ]:
ct_plot = per_ct_df.sort_values('OR_HAG_vs_NAG', ascending=True).copy()
y_pos   = np.arange(len(ct_plot))

colors_bar = [
    '#E05C2A' if p < 0.05 else ('#F5A07A' if p < 0.10 else '#DDDDDD')
    for p in ct_plot['p_adj_HAG_vs_NAG']
]

fig, ax = plt.subplots(figsize=(10, 7))
ax.barh(y_pos, ct_plot['OR_HAG_vs_NAG'], color=colors_bar, edgecolor='white', height=0.7)
ax.axvline(1.0, color='black', linewidth=1.0, linestyle='--')

x_max = ct_plot['OR_HAG_vs_NAG'].max()
ax.set_xlim(0, x_max * 1.85)

ax.set_yticks(y_pos)
ax.set_yticklabels(ct_plot['cell_type'], fontsize=9)
ax.set_xlabel('Odds ratio (HAG vs NAG) for top-25 marker status', fontsize=11)
ax.set_title(
    'Panel D: Per-neuronal-cell-type HAG marker enrichment\n'
    '(top-25 FC markers per cell type, two-sided Fisher, BH-FDR)',
    fontsize=11, fontweight='bold'
)

# Gene annotations for OR > 2
for i, (_, row) in enumerate(ct_plot.iterrows()):
    if row['OR_HAG_vs_NAG'] > 2.0 and row['top_HAG_genes']:
        ax.text(row['OR_HAG_vs_NAG'] + 0.15, i, row['top_HAG_genes'],
                va='center', fontsize=7.5, color='#333333')

patches = [
    mpatches.Patch(color='#E05C2A', label='FDR < 0.05'),
    mpatches.Patch(color='#F5A07A', label='FDR < 0.10'),
    mpatches.Patch(color='#DDDDDD', label='ns'),
]
ax.legend(handles=patches, fontsize=9, loc='lower right')
sns.despine(ax=ax)
plt.tight_layout()
plt.savefig('/mnt/results/panel_D_only.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved panel_D_only.png')